# DS3/GSE182109 — build the merged raw-count AnnData object

This notebook reconstructs the GSE182109 analysis object from the 44
sample-level 10x Genomics matrix triplets extracted from the GEO RAW
archive.

Every sample must have:

- `<prefix>_matrix.mtx.gz`
- `<prefix>_features.tsv.gz`
- `<prefix>_barcodes.tsv.gz`

The prefix encodes the GEO sample and biological label, for example:

`GSM5518596_rGBM-01-A`

The original pipeline used gene symbols as the feature axis, made duplicate
symbols unique within each sample, appended the sample prefix to each cell
barcode, and concatenated samples with an outer gene join.

## Expected merged object

- 44 samples
- 264,951 cells
- 38,224 genes

## Default paths

Input:

`data/raw/GSE182109/`

Output:

`data/interim/DS3_GSE182109/GSE182109_merged_rawcounts.h5ad`

Override paths with `GLIOMA_PROJECT_ROOT`, `GSE182109_RAW_DIR`, and
`GSE182109_MERGED_H5AD`.

In [ ]:
import gc
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

RAW_DIR = Path(
    os.environ.get(
        "GSE182109_RAW_DIR",
        PROJECT_ROOT / "data" / "raw" / "GSE182109",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE182109_MERGED_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS3_GSE182109"
        / "GSE182109_merged_rawcounts.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS3_GSE182109" / "00a_build"
)

EXPECTED_N_SAMPLES = 44
EXPECTED_OUTPUT_SHAPE = (264_951, 38_224)

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAW directory:", RAW_DIR)
print("Output:", OUTPUT_H5AD)

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"GSE182109 RAW directory not found: {RAW_DIR}\n"
        "Extract the 10x files there or set GSE182109_RAW_DIR."
    )

In [ ]:
PREFIX_PATTERN = re.compile(
    r"^(GSM\d+)_(LGG|ndGBM|rGBM)-([^-]+)(?:-(.+))?$"
)


def parse_sample_metadata(prefix: str) -> dict:
    match = PREFIX_PATTERN.match(prefix)
    if match is None:
        raise ValueError(
            f"Unexpected GSE182109 sample prefix: {prefix!r}. "
            "Expected GSM..._<LGG|ndGBM|rGBM>-<patient>[-<region>]."
        )

    gsm, group, patient, region = match.groups()
    return {
        "gsm": gsm,
        "sample": prefix,
        "sample_id": prefix,
        "group": group,
        "patient": patient,
        "region": region if region is not None else "",
        "sample_key": prefix,
        "geo_id": "GSE182109",
        "core_dataset": "DS3_GSE182109",
    }


def discover_complete_10x_triplets(raw_dir: Path) -> list[dict]:
    matrix_paths = sorted(raw_dir.glob("GSM*_matrix.mtx.gz"))
    if not matrix_paths:
        raise FileNotFoundError(
            f"No files matching GSM*_matrix.mtx.gz were found in {raw_dir}"
        )

    records = []
    for matrix_path in matrix_paths:
        prefix = matrix_path.name.removesuffix("_matrix.mtx.gz")
        features_path = raw_dir / f"{prefix}_features.tsv.gz"
        barcodes_path = raw_dir / f"{prefix}_barcodes.tsv.gz"

        missing = [
            path.name
            for path in (features_path, barcodes_path)
            if not path.exists()
        ]
        if missing:
            raise FileNotFoundError(
                f"Incomplete 10x triplet for {prefix}: missing {missing}"
            )

        metadata = parse_sample_metadata(prefix)
        records.append(
            {
                "prefix": prefix,
                "matrix_path": matrix_path,
                "features_path": features_path,
                "barcodes_path": barcodes_path,
                **metadata,
            }
        )

    if len(records) != EXPECTED_N_SAMPLES:
        raise AssertionError(
            f"Expected {EXPECTED_N_SAMPLES} complete sample triplets, "
            f"observed {len(records)}."
        )

    return records


input_records = discover_complete_10x_triplets(RAW_DIR)

inventory = pd.DataFrame(
    [
        {
            "prefix": record["prefix"],
            "gsm": record["gsm"],
            "group": record["group"],
            "patient": record["patient"],
            "region": record["region"],
            "matrix_file": record["matrix_path"].name,
            "features_file": record["features_path"].name,
            "barcodes_file": record["barcodes_path"].name,
        }
        for record in input_records
    ]
)
inventory.to_csv(
    AUDIT_DIR / "GSE182109_input_10x_inventory.tsv",
    sep="\t",
    index=False,
)
inventory

In [ ]:
def matrix_values(matrix):
    return matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()


def assert_raw_counts(matrix, label: str):
    values = matrix_values(matrix)
    if values.size == 0:
        raise ValueError(f"{label} has no nonzero values.")
    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")
    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")
    if not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} is not an integer raw-count matrix.")


def read_sample(record: dict) -> ad.AnnData:
    sample = sc.read_10x_mtx(
        RAW_DIR,
        prefix=record["prefix"] + "_",
        var_names="gene_symbols",
        make_unique=False,
        cache=False,
        gex_only=True,
    )

    if sample.n_obs == 0 or sample.n_vars == 0:
        raise ValueError(
            f"{record['prefix']}: the parsed 10x object is empty."
        )

    original_symbols = sample.var_names.astype(str)
    sample.var["feature_name_original"] = original_symbols
    sample.var_names_make_unique()

    if not sample.var_names.is_unique:
        raise AssertionError(
            f"{record['prefix']}: gene symbols remain duplicated."
        )

    if sp.issparse(sample.X):
        sample.X = sample.X.tocsr().astype(np.float32, copy=False)
    else:
        sample.X = sp.csr_matrix(
            np.asarray(sample.X, dtype=np.float32)
        )

    assert_raw_counts(sample.X, f"{record['prefix']}: adata.X")

    original_barcodes = sample.obs_names.astype(str)
    sample.obs["cell_barcode"] = original_barcodes
    sample.obs_names = pd.Index(
        [
            f"{barcode}-{record['prefix']}"
            for barcode in original_barcodes
        ],
        name="cell_id",
    )

    if not sample.obs_names.is_unique:
        raise AssertionError(
            f"{record['prefix']}: cell identifiers are not unique."
        )

    for key in (
        "gsm",
        "sample",
        "sample_id",
        "group",
        "patient",
        "region",
        "sample_key",
        "geo_id",
        "core_dataset",
    ):
        sample.obs[key] = record[key]

    sample.uns["source_prefix"] = record["prefix"]
    sample.uns["source_geo_accession"] = record["gsm"]
    return sample


sample_objects = []
sample_summary_rows = []
var_frames = []

for index, record in enumerate(input_records, start=1):
    print(
        f"[{index}/{len(input_records)}] Loading "
        f"{record['prefix']}"
    )
    sample = read_sample(record)

    sample_summary_rows.append(
        {
            "gsm": record["gsm"],
            "sample": record["sample"],
            "group": record["group"],
            "patient": record["patient"],
            "region": record["region"],
            "cells": int(sample.n_obs),
            "genes": int(sample.n_vars),
            "total_counts": int(sample.X.sum()),
            "maximum_count": int(sample.X.max()),
        }
    )

    var_frame = sample.var.copy()
    var_frame["var_name"] = sample.var_names.astype(str)
    var_frame["source_sample"] = record["sample"]
    var_frames.append(var_frame.reset_index(drop=True))

    sample_objects.append(sample)
    print(sample)
    gc.collect()

sample_summary = pd.DataFrame(sample_summary_rows)
sample_summary.to_csv(
    AUDIT_DIR / "GSE182109_raw_build_sample_summary.tsv",
    sep="\t",
    index=False,
)
sample_summary

In [ ]:
adata_merged = ad.concat(
    sample_objects,
    axis="obs",
    join="outer",
    merge="first",
    index_unique=None,
    fill_value=0,
    pairwise=False,
)

if not adata_merged.obs_names.is_unique:
    raise AssertionError("Merged cell identifiers are not unique.")
if adata_merged.shape != EXPECTED_OUTPUT_SHAPE:
    raise AssertionError(
        "The GSE182109 merge did not reproduce the historical object. "
        f"Expected {EXPECTED_OUTPUT_SHAPE}, observed {adata_merged.shape}. "
        "Inspect the sample inventory and per-sample summary."
    )
if not sp.issparse(adata_merged.X):
    adata_merged.X = sp.csr_matrix(adata_merged.X)
else:
    adata_merged.X = adata_merged.X.tocsr()

assert_raw_counts(adata_merged.X, "merged adata.X")

print("Merged object:", adata_merged)
print(adata_merged.obs["group"].value_counts())

In [ ]:
# Reconstruct a complete feature-metadata table across the outer gene union.
# Conflicting annotations are audited; the first non-empty value is retained.
all_var = pd.concat(var_frames, ignore_index=True)

metadata_columns = [
    column
    for column in ("gene_ids", "feature_types", "feature_name_original")
    if column in all_var.columns
]

conflict_rows = []
consolidated = pd.DataFrame(index=adata_merged.var_names)

for column in metadata_columns:
    values_by_gene = (
        all_var[["var_name", column]]
        .dropna()
        .assign(**{column: lambda frame: frame[column].astype(str)})
    )
    values_by_gene = values_by_gene[
        values_by_gene[column].str.strip() != ""
    ]

    grouped = values_by_gene.groupby("var_name", sort=False)[column]
    consolidated[column] = grouped.first().reindex(
        adata_merged.var_names
    )

    nunique = grouped.nunique()
    conflicting_genes = nunique[nunique > 1].index
    for gene in conflicting_genes:
        observed_values = sorted(
            values_by_gene.loc[
                values_by_gene["var_name"] == gene,
                column,
            ].unique().tolist()
        )
        conflict_rows.append(
            {
                "var_name": gene,
                "field": column,
                "observed_values": " | ".join(observed_values),
            }
        )

# Preserve any metadata already populated by AnnData concat.
for column in adata_merged.var.columns:
    if column not in consolidated:
        consolidated[column] = adata_merged.var[column]

adata_merged.var = consolidated

conflict_table = pd.DataFrame(
    conflict_rows,
    columns=["var_name", "field", "observed_values"],
)
conflict_table.to_csv(
    AUDIT_DIR / "GSE182109_feature_metadata_conflicts.tsv",
    sep="\t",
    index=False,
)

adata_merged.uns["dataset_id"] = "DS3_GSE182109"
adata_merged.uns["geo_accession"] = "GSE182109"
adata_merged.uns["build_timestamp"] = datetime.now().isoformat(
    timespec="seconds"
)
adata_merged.uns["source_sample_count"] = EXPECTED_N_SAMPLES
adata_merged.uns["feature_join"] = "outer"
adata_merged.uns["feature_metadata_conflict_count"] = int(
    conflict_table.shape[0]
)

build_summary = {
    "dataset_id": "DS3_GSE182109",
    "geo_accession": "GSE182109",
    "n_samples": int(adata_merged.obs["sample"].nunique()),
    "n_cells": int(adata_merged.n_obs),
    "n_genes": int(adata_merged.n_vars),
    "matrix_format": type(adata_merged.X).__name__,
    "matrix_dtype": str(adata_merged.X.dtype),
    "feature_metadata_conflicts": int(conflict_table.shape[0]),
    "output_h5ad": str(OUTPUT_H5AD),
}

with open(
    AUDIT_DIR / "GSE182109_raw_build_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(build_summary, handle, indent=2)

print(build_summary)

In [ ]:
temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata_merged.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

with sc.read_h5ad(OUTPUT_H5AD, backed="r") as check:
    if check.shape != EXPECTED_OUTPUT_SHAPE:
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )
    for column in (
        "gsm",
        "sample",
        "sample_id",
        "group",
        "patient",
        "region",
        "sample_key",
        "cell_barcode",
    ):
        if column not in check.obs:
            raise AssertionError(
                f"Saved output is missing adata.obs[{column!r}]."
            )

print("Saved and validated:", OUTPUT_H5AD)